# 01 — From n-grams to RNN, LSTM, and GRU

    **Companion chapter:** `01-language-models.md`

    ## Learning goals

    - Estimate n-gram probabilities from counts.
- Observe sparsity and smoothing.
- Trace a recurrent hidden state.
- Train a tiny word-level recurrent language model.
- Swap RNN, LSTM, and GRU modules.

    ## How to use this notebook

    Run the cells from top to bottom. Read the comments, change small values, and
    rerun the cell. Every notebook ends with practice prompts that can become
    GitHub issues, exercises, or discussion questions.

## 1. A tiny corpus

We use a deliberately small corpus so every count and tensor can be inspected.
The tokenizer below is whitespace-based. It is suitable for teaching, but a
real Persian project should use a proper normalization and tokenization pipeline.

In [1]:
from __future__ import annotations

import math
import random
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [2]:
corpus = [
    "من شعر فارسی را دوست دارم",
    "من کتاب فارسی را می خوانم",
    "سارا شعر نو را دوست دارد",
    "علی کتاب زبان را می خواند",
    "من چای را دوست دارم",
]

tokenized = [sentence.split() for sentence in corpus]
tokenized

[['من', 'شعر', 'فارسی', 'را', 'دوست', 'دارم'],
 ['من', 'کتاب', 'فارسی', 'را', 'می', 'خوانم'],
 ['سارا', 'شعر', 'نو', 'را', 'دوست', 'دارد'],
 ['علی', 'کتاب', 'زبان', 'را', 'می', 'خواند'],
 ['من', 'چای', 'را', 'دوست', 'دارم']]

## 2. Build n-gram counts

An n-gram model predicts a token from a fixed history. The helper adds boundary
tokens so that sentence beginnings and endings are also learned.

In [3]:
def add_boundaries(tokens: list[str], n: int) -> list[str]:
    return ["<BOS>"] * (n - 1) + tokens + ["<EOS>"]


def ngram_counts(sentences: list[list[str]], n: int):
    context_counts = Counter()
    ngram_counts_ = Counter()

    for sentence in sentences:
        padded = add_boundaries(sentence, n)
        for i in range(n - 1, len(padded)):
            context = tuple(padded[i - n + 1 : i])
            token = padded[i]
            context_counts[context] += 1
            ngram_counts_[context + (token,)] += 1

    return context_counts, ngram_counts_


bigram_contexts, bigrams = ngram_counts(tokenized, n=2)
trigram_contexts, trigrams = ngram_counts(tokenized, n=3)

print("Bigram examples:")
for item, count in bigrams.most_common(8):
    print(item, "->", count)

Bigram examples:
('<BOS>', 'من') -> 3
('را', 'دوست') -> 3
('فارسی', 'را') -> 2
('دوست', 'دارم') -> 2
('دارم', '<EOS>') -> 2
('را', 'می') -> 2
('من', 'شعر') -> 1
('شعر', 'فارسی') -> 1


## 3. Next-token probabilities with add-one smoothing

Raw counts assign probability zero to unseen n-grams. Add-one smoothing is not
the strongest smoothing method, but it makes the sparsity problem visible.

In [4]:
vocabulary = sorted({token for sentence in tokenized for token in sentence} | {"<EOS>"})
vocab_size = len(vocabulary)


def smoothed_probability(
    context: tuple[str, ...],
    token: str,
    context_counts: Counter,
    full_counts: Counter,
    alpha: float = 1.0,
) -> float:
    numerator = full_counts[context + (token,)] + alpha
    denominator = context_counts[context] + alpha * vocab_size
    return numerator / denominator


context = ("دوست",)
distribution = {
    token: smoothed_probability(context, token, bigram_contexts, bigrams)
    for token in vocabulary
}

sorted(distribution.items(), key=lambda item: item[1], reverse=True)[:8]

[('دارم', 0.15),
 ('دارد', 0.1),
 ('<EOS>', 0.05),
 ('خواند', 0.05),
 ('خوانم', 0.05),
 ('دوست', 0.05),
 ('را', 0.05),
 ('زبان', 0.05)]

In [5]:
def sentence_log_probability(
    tokens: list[str],
    n: int,
    context_counts: Counter,
    full_counts: Counter,
    alpha: float = 1.0,
) -> float:
    padded = add_boundaries(tokens, n)
    total = 0.0

    for i in range(n - 1, len(padded)):
        context = tuple(padded[i - n + 1 : i])
        token = padded[i]
        probability = smoothed_probability(
            context, token, context_counts, full_counts, alpha
        )
        total += math.log(probability)

    return total


test_sentence = "من شعر فارسی را دوست دارم".split()
log_p = sentence_log_probability(
    test_sentence, 2, bigram_contexts, bigrams, alpha=1.0
)
perplexity = math.exp(-log_p / (len(test_sentence) + 1))

print("Log probability:", round(log_p, 3))
print("Perplexity:", round(perplexity, 3))

Log probability: -13.552
Perplexity: 6.931


## 4. An RNN state by hand

This reproduces the scalar recurrence idea: the current state combines the new
input with the previous state.

In [6]:
inputs = [1.0, 0.5, -0.2]
h = 0.0
states = []

for x_t in inputs:
    h = np.tanh(x_t + 0.5 * h)
    states.append(h)

pd.DataFrame({"x_t": inputs, "h_t": states})

,x_t,h_t
0,1.0,0.761594
1,0.5,0.706818
2,-0.2,0.152217


## 5. Prepare word-level training data

We now train a tiny recurrent language model. The same class can use an ordinary
RNN, an LSTM, or a GRU.

In [7]:
special_tokens = ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]
words = sorted({word for sentence in tokenized for word in sentence})
itos = special_tokens + words
stoi = {token: index for index, token in enumerate(itos)}

PAD_ID = stoi["<PAD>"]
BOS_ID = stoi["<BOS>"]
EOS_ID = stoi["<EOS>"]


def encode(sentence: list[str]) -> list[int]:
    return [BOS_ID] + [stoi.get(token, stoi["<UNK>"]) for token in sentence] + [EOS_ID]


encoded = [torch.tensor(encode(sentence), dtype=torch.long) for sentence in tokenized]
inputs = nn.utils.rnn.pad_sequence(
    [sequence[:-1] for sequence in encoded],
    batch_first=True,
    padding_value=PAD_ID,
)
targets = nn.utils.rnn.pad_sequence(
    [sequence[1:] for sequence in encoded],
    batch_first=True,
    padding_value=PAD_ID,
)

print("Input shape:", tuple(inputs.shape))
print("Target shape:", tuple(targets.shape))
print("First input:", [itos[i] for i in inputs[0].tolist()])

Input shape: (5, 7)
Target shape: (5, 7)
First input: ['<BOS>', 'من', 'شعر', 'فارسی', 'را', 'دوست', 'دارم']


In [8]:
class TinyRecurrentLM(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int = 24,
        hidden_dim: int = 48,
        cell_type: str = "gru",
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_ID)

        recurrent_classes = {
            "rnn": nn.RNN,
            "lstm": nn.LSTM,
            "gru": nn.GRU,
        }
        if cell_type not in recurrent_classes:
            raise ValueError(f"cell_type must be one of {list(recurrent_classes)}")

        self.recurrent = recurrent_classes[cell_type](
            embedding_dim,
            hidden_dim,
            batch_first=True,
        )
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(token_ids)
        hidden_states, _ = self.recurrent(embedded)
        return self.output(hidden_states)


for cell_type in ["rnn", "lstm", "gru"]:
    model = TinyRecurrentLM(len(itos), cell_type=cell_type)
    logits = model(inputs)
    parameters = sum(parameter.numel() for parameter in model.parameters())
    print(f"{cell_type.upper():4s} logits={tuple(logits.shape)}, parameters={parameters:,}")

RNN  logits=(5, 7, 20), parameters=5,012
LSTM logits=(5, 7, 20), parameters=15,668
GRU  logits=(5, 7, 20), parameters=12,116


## 6. Train the GRU version

The goal is not a useful production model. It is to connect token IDs,
embeddings, recurrent states, vocabulary logits, and cross-entropy loss.

In [9]:
model = TinyRecurrentLM(len(itos), cell_type="gru").to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
loss_function = nn.CrossEntropyLoss(ignore_index=PAD_ID)

x_train = inputs.to(device)
y_train = targets.to(device)

history = []
for epoch in range(80):
    model.train()
    optimizer.zero_grad()

    logits = model(x_train)
    loss = loss_function(logits.reshape(-1, len(itos)), y_train.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    history.append(loss.item())

print("Initial loss:", round(history[0], 3))
print("Final loss:", round(history[-1], 3))

plt.plot(history)
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Tiny GRU language model")
plt.show()

Initial loss: 3.058
Final loss: 0.237


/tmp/ipykernel_557/1895747915.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
@torch.no_grad()
def generate(model: nn.Module, prefix: list[str], max_new_tokens: int = 8) -> list[str]:
    model.eval()
    generated = [BOS_ID] + [stoi.get(token, stoi["<UNK>"]) for token in prefix]

    for _ in range(max_new_tokens):
        x = torch.tensor([generated], dtype=torch.long, device=device)
        next_logits = model(x)[0, -1]
        next_id = int(next_logits.argmax())

        if next_id == EOS_ID:
            break
        generated.append(next_id)

    return [itos[index] for index in generated[1:]]


print(generate(model, ["من"]))
print(generate(model, ["سارا"]))

['من', 'چای', 'را', 'دوست', 'دارم']
['سارا', 'شعر', 'نو', 'را', 'دوست', 'دارد']


## Practice

1. Change `n=2` to `n=3` and compare sentence perplexities.
2. Remove one sentence from the corpus. Which n-grams become unseen?
3. Train `cell_type="rnn"` and compare its loss curve with the GRU.
4. Add a longer sentence containing a long-distance subject–verb dependency.
5. Replace whitespace tokenization with your Persian preprocessing pipeline.